In [ ]:
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

In [1]:
import numpy as np
import pandas as pd
import os
import gc
from sklearn.preprocessing import OrdinalEncoder


# =====================================================================
# STEP 1: OPTIMIZED DATA LOADING & STRUCTURAL LEFT-MERGE
# =====================================================================

print("Loading datasets...")
# Using a left join to retain 100% of transactions. Missing identity 
# records are naturally preserved as structural null values.
BASE_DIR = '/Users/abannee/Documents/GitHub/fraud_detection_ml/'
DATA_DIR = os.path.join(BASE_DIR, 'data/raw')

print("Loading Train Datasets...")
train_transaction = pd.read_csv(os.path.join(DATA_DIR, 'train_transaction.csv'))
train_identity = pd.read_csv(os.path.join(DATA_DIR, "train_identity.csv"))

print("Loading Test Datasets...")
test_transaction = pd.read_csv(os.path.join(DATA_DIR, 'test_transaction.csv'))
test_identity = pd.read_csv(os.path.join(DATA_DIR, "test_identity.csv"))


print("Merging transaction and identity data on TransactionID...")
train_df = pd.merge(train_transaction, train_identity, on="TransactionID", how="left")
# Clean up memory allocation from unmerged frames
del train_transaction, train_identity
# Force immediate background RAM reclamation
gc.collect() 

print("Merging Test tables (Left Join)...")
test_df = pd.merge(test_transaction, test_identity, on="TransactionID", how="left")
# Clean up memory allocation from unmerged frames
del test_transaction, test_identity
gc.collect()

print(f"Data loading complete. Train shape: {train_df.shape}, Test shape: {test_df.shape}")

Loading datasets...
Loading Train Datasets...
Loading Test Datasets...
Merging transaction and identity data on TransactionID...
Merging Test tables (Left Join)...
Data loading complete. Train shape: (590540, 434), Test shape: (506691, 433)


In [16]:
# =====================================================================
# STEP 2: CHRONOLOGICAL VALIDATION SPLIT (70% Train / 30% Local Test)
# =====================================================================

# 1. Enforce strict chronological order before cutting
train_df = train_df.sort_values(by="TransactionDT").reset_index(drop=True)

# 2. Calculate the deterministic split point index
# 2. Define a local split point (e.g., use the first 70% of train_df for training, last 30% for validation)
split_idx = int(len(train_df) * 0.70)


print(f"Splitting train_df at index: {split_idx}")
print(f"-> Training Pool: Rows 0 to {split_idx - 1}")
print(f"-> Validation Pool: Rows {split_idx} to {len(train_df) - 1}\n")

# 3. Slice features and targets out of the 70% Training Portion
X_train = train_df.iloc[:split_idx].drop(columns=["isFraud"])
y_train = train_df.iloc[:split_idx]["isFraud"]

# 4. Slice features and targets out of the 30% Validation Portion (Your Local Test)
X_val = train_df.iloc[split_idx:].drop(columns=["isFraud"])
y_val = train_df.iloc[split_idx:]["isFraud"]

print("--- Final Validation Setup Arrays ---")
print(f"X_train Shape (Features) : {X_train.shape} | y_train Shape (Labels): {y_train.shape}")
print(f"X_val Shape (Local Test) : {X_val.shape}  | y_val Shape (Labels): {y_val.shape}")

# 5. Handle the Kaggle competition test set cleanly
# We preserve this for our final submission predictions
X_test_submission = test_df.copy()
print(f"X_test_submission Shape  : {X_test_submission.shape}")

Splitting train_df at index: 413378
-> Training Pool: Rows 0 to 413377
-> Validation Pool: Rows 413378 to 590539

--- Final Validation Setup Arrays ---
X_train Shape (Features) : (413378, 433) | y_train Shape (Labels): (413378,)
X_val Shape (Local Test) : (177162, 433)  | y_val Shape (Labels): (177162,)
X_test_submission Shape  : (506691, 433)


In [28]:
X_test_submission.columns

Index(['TransactionID', 'TransactionDT', 'TransactionAmt', 'ProductCD',
       'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
       ...
       'id-31', 'id-32', 'id-33', 'id-34', 'id-35', 'id-36', 'id-37', 'id-38',
       'DeviceType', 'DeviceInfo'],
      dtype='str', length=433)

In [23]:
X_train[train_avail_cols]

,ProductCD,card4,id_30,id_31,DeviceType
0,4.0,1.0,NaN,NaN,NaN
1,4.0,2.0,NaN,NaN,NaN
2,4.0,3.0,NaN,NaN,NaN
3,4.0,2.0,NaN,NaN,NaN
4,1.0,2.0,64.0,42.0,1.0
...,...,...,...,...,...
413373,4.0,3.0,NaN,NaN,NaN
413374,4.0,3.0,NaN,NaN,NaN
413375,4.0,3.0,NaN,NaN,NaN
413376,4.0,2.0,NaN,NaN,NaN


In [20]:
requested_categorical_cols

['ProductCD', 'card4', 'id_30', 'id_31', 'DeviceType']

In [27]:
X_test_submission.columns

Index(['TransactionID', 'TransactionDT', 'TransactionAmt', 'ProductCD',
       'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
       ...
       'id-31', 'id-32', 'id-33', 'id-34', 'id-35', 'id-36', 'id-37', 'id-38',
       'DeviceType', 'DeviceInfo'],
      dtype='str', length=433)

In [21]:
test_avail_cols

['ProductCD', 'card4', 'DeviceType']

In [19]:
import pandas as pd
import numpy as np
import gc
from sklearn.preprocessing import OrdinalEncoder

# =====================================================================
# STEP 3: DEFENSIVE CATEGORICAL ENCODING (MULTI-SUBSET SAFE)
# =====================================================================

# Define our ideal global target list of categorical columns
requested_categorical_cols = ["ProductCD", "card4", "id_30", "id_31", "DeviceType"]

# --- A. Safe Label (Ordinal) Encoding ---
print("Executing safe Ordinal Encoding...")

# Configure encoder defensively to route unseen labels to a fallback value (-1)
ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

# 1. Determine columns available strictly in the Training set to fit the encoder
train_avail_cols = [col for col in requested_categorical_cols if col in X_train.columns]
print(f"-> Fitting encoder on columns present in X_train: {train_avail_cols}")

# Fit strictly on training data
X_train[train_avail_cols] = ordinal_encoder.fit_transform(X_train[train_avail_cols].astype(str))

# 2. Transform validation set safely based on what it actually contains
val_avail_cols = [col for col in requested_categorical_cols if col in X_val.columns]
if val_avail_cols:
    X_val[val_avail_cols] = ordinal_encoder.transform(X_val[val_avail_cols].astype(str))

# 3. Transform test submission safely based on what it actually contains (Prevents KeyError)
test_avail_cols = [col for col in requested_categorical_cols if col in X_test_submission.columns]
if test_avail_cols:
    X_test_submission[test_avail_cols] = ordinal_encoder.transform(X_test_submission[test_avail_cols].astype(str))
else:
    print("Warning: None of the target categorical columns found in X_test_submission!")


# --- B. Safe Target Encoding (m-estimate Smoothing Pipeline) ---
print("\nExecuting safe Target Encoding pipeline...")

# Calculate global training baseline fraud rate to act as a neutral anchor
global_training_mean = y_train.mean()
print(f"Global Training Baseline Fraud Rate: {global_training_mean:.5f}")

# Target encode columns that exist in the training data
for col in train_avail_cols:
    train_labels = X_train[col].astype(str)
    
    # Calculate group totals strictly inside training records
    stats = pd.DataFrame({"target": y_train, "label": train_labels}).groupby("label")
    category_means = stats["target"].mean()
    category_counts = stats["target"].count()
    
    # Apply m-estimate smoothing formula
    m = 10 
    smoothed_vals = (category_means * category_counts + global_training_mean * m) / (category_counts + m)
    encoding_map = smoothed_vals.to_dict()
    
    # Map back to X_train
    X_train[f"{col}_target_enc"] = train_labels.map(encoding_map).fillna(global_training_mean)
    
    # Map back to X_val if the column exists there
    if col in X_val.columns:
        X_val[f"{col}_target_enc"] = X_val[col].astype(str).map(encoding_map).fillna(global_training_mean)
        
    # Map back to X_test_submission if the column exists there
    if col in X_test_submission.columns:
        X_test_submission[f"{col}_target_enc"] = X_test_submission[col].astype(str).map(encoding_map).fillna(global_training_mean)
    else:
        # Critical strategy: If test is missing the root feature entirely, seed it with the baseline mean column
        # This keeps feature dimensions perfectly aligned across sets for XGBoost/LightGBM down the line.
        X_test_submission[f"{col}_target_enc"] = global_training_mean
        print(f"-> {col} missing from test submission. Created target_enc placeholder filled with global baseline.")

# Final garbage collection wipe
gc.collect()

print("\nData preparation pipeline complete!")
print(f"-> Train Features Shape: {X_train.shape} | Train Labels Shape: {y_train.shape}")
print(f"-> Val Features Shape  : {X_val.shape}  | Val Labels Shape  : {y_val.shape}")
print(f"-> Test Sub Shape      : {X_test_submission.shape}")

Executing safe Ordinal Encoding...
-> Fitting encoder on columns present in X_train: ['ProductCD', 'card4', 'id_30', 'id_31', 'DeviceType']


ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- id_30
- id_31


###### =====================================================================
#### STEP 1: OPTIMIZED DATA LOADING & STRUCTURAL LEFT-MERGE
###### =====================================================================

In [29]:
import numpy as np
import pandas as pd
import os
import gc
from sklearn.preprocessing import OrdinalEncoder


print("Loading datasets...")
# Using a left join to retain 100% of transactions. Missing identity 
# records are naturally preserved as structural null values.
BASE_DIR = '/Users/abannee/Documents/GitHub/fraud_detection_ml/'
DATA_DIR = os.path.join(BASE_DIR, 'data/raw')

print("Loading Train Datasets...")
train_transaction = pd.read_csv(os.path.join(DATA_DIR, 'train_transaction.csv'))
train_identity = pd.read_csv(os.path.join(DATA_DIR, "train_identity.csv"))

print("Merging transaction and identity data on TransactionID...")
df = pd.merge(train_transaction, train_identity, on="TransactionID", how="left")

# Clean up memory allocation from unmerged frames
del train_transaction, train_identity

# Force immediate background RAM reclamation
gc.collect() 


print(f"Data loading complete. data shape: {df.shape}")

Loading datasets...
Loading Train Datasets...
Merging transaction and identity data on TransactionID...
Data loading complete. data shape: (590540, 434)


###### =====================================================================
#### STEP 2: CHRONOLOGICAL SORTING & TIME-BASED SPLITTING (70/30)
###### =====================================================================

In [30]:
print("Sorting data chronologically by TransactionDT to prevent look-ahead bias...")
df = df.sort_values(by="TransactionDT").reset_index(drop=True)

# Calculate deterministic, shuffle-free split index for OOT validation
split_idx = int(len(df) * 0.70)

print(# Using a left join to retain 100% of transactions. Missing identity 
# records are naturally preserved as structural null values.
f"Splitting data: Training on first 70% ({split_idx} rows), Testing on remaining 30% ({len(df) - split_idx} rows)...")
train_df = df.iloc[:split_idx].reset_index(drop=True)
test_df = df.iloc[split_idx:].reset_index(drop=True)

# Separate features and target
X_train = train_df.drop(columns=["isFraud"])
y_train = train_df["isFraud"]
X_test = test_df.drop(columns=["isFraud"])
y_test = test_df["isFraud"]

Sorting data chronologically by TransactionDT to prevent look-ahead bias...
Splitting data: Training on first 70% (413378 rows), Testing on remaining 30% (177162 rows)...


###### =====================================================================
#### STEP 3: DEFENSIVE CATEGORICAL ENCODING
###### =====================================================================

In [31]:
# Define high-cardinality and operational categorical columns to encode
categorical_cols = ["ProductCD", "card4", "id_30", "id_31", "DeviceType"]
# Ensure columns exist in the dataframe before processing
categorical_cols = [col for col in categorical_cols if col in X_train.columns]

# --- A. Safe Label (Ordinal) Encoding ---
print("Executing safe Ordinal Encoding...")
# Configure encoder defensively to route unseen production/test labels to a fallback value (-1)
ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

# Fit strictly on training data, transform both partitions
X_train[categorical_cols] = ordinal_encoder.fit_transform(X_train[categorical_cols].astype(str))
X_test[categorical_cols] = ordinal_encoder.transform(X_test[categorical_cols].astype(str))


# --- B. Safe Target Encoding (Out-of-Fold / Smoothed Mock) ---
print("Executing safe Target Encoding pipeline...")

# Calculate global training baseline fraud rate to act as a neutral anchor
global_training_mean = y_train.mean()

for col in categorical_cols:
    # Example Target Encoding Strategy with local calculation.
    # To prevent target leakage completely, we calculate map dictionaries 
    # strictly from training pairs, incorporating a basic frequency map.
    
    # Calculate category stats within training set
    stats = pd.DataFrame({"target": y_train, "label": train_df[col]}).groupby("label")
    category_means = stats["target"].mean()
    category_counts = stats["target"].count()
    
    # Simple m-estimate smoothing formula: (mean * count + global_mean * m) / (count + m)
    m = 10 
    smoothed_vals = (category_means * category_counts + global_training_mean * m) / (category_counts + m)
    encoding_map = smoothed_vals.to_dict()
    
    # Map back to train and test data using training metrics only
    # Defensively fill any novel/unseen categories in test with the neutral global training mean
    X_train[f"{col}_target_enc"] = train_df[col].map(encoding_map).fillna(global_training_mean)
    X_test[f"{col}_target_enc"] = test_df[col].map(encoding_map).fillna(global_training_mean)

print("Data preparation pipeline complete! Ready for feature engineering or modeling steps.")

Executing safe Ordinal Encoding...
Executing safe Target Encoding pipeline...


/var/folders/2w/c8v4yt5j21s3nkv8j9x_fc340000gn/T/ipykernel_61136/3234097378.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train[f"{col}_target_enc"] = train_df[col].map(encoding_map).fillna(global_training_mean)
/var/folders/2w/c8v4yt5j21s3nkv8j9x_fc340000gn/T/ipykernel_61136/3234097378.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_test[f"{col}_target_enc"] = test_df[col].map(encoding_map).fillna(global_training_mean)
/var/folders/2w/c8v4yt5j21s3nkv8j9x_fc340000gn/T/ipykernel_61136/3234097378.py:39: Performance

Data preparation pipeline complete! Ready for feature engineering or modeling steps.


/var/folders/2w/c8v4yt5j21s3nkv8j9x_fc340000gn/T/ipykernel_61136/3234097378.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train[f"{col}_target_enc"] = train_df[col].map(encoding_map).fillna(global_training_mean)
/var/folders/2w/c8v4yt5j21s3nkv8j9x_fc340000gn/T/ipykernel_61136/3234097378.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_test[f"{col}_target_enc"] = test_df[col].map(encoding_map).fillna(global_training_mean)
